# 01 — Data Collection
CARE Dashboard — Climate Awareness and Risk Evaluation

In [1]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


## Path configuration and dataset availability check

In [ ]:
from pathlib import Path



BASE = str((Path.cwd() / ".." / "002_Dataset").resolve())

SEPA_PATH        = f"{BASE}/raw/sepa/PVAv2.gpkg"
OSM_BUILD        = f"{BASE}/raw/osm/osm_buildings_glasgow.gpkg"
OSM_ROADS        = f"{BASE}/raw/osm/osm_roads_glasgow.gpkg"
OSM_WATER        = f"{BASE}/raw/osm/osm_water_glasgow.gpkg"
NASA_ELEV        = f"{BASE}/raw/nasa_elevation/nasa_elevation_glasgow.gpkg"
RAIN_ARCHIVE     = f"{BASE}/raw/rainfall"                      # 1987-2025, 467 files (merged)
RAIN_DAILY_PARQUET = f"{BASE}/processed/rainfall_daily_1987_2025.parquet"
RAIN_FEATURES_40YR = f"{BASE}/processed/rainfall_features_40yr.csv"
FEATURE_CSV      = f"{BASE}/processed/feature_matrix.csv"
FEATURE_CSV_COORDS = f"{BASE}/processed/feature_matrix_with_coords.csv"
FEATURE_CSV_40YR = f"{BASE}/processed/feature_matrix_40yr.csv"
FEATURE_CSV_3YR  = f"{BASE}/processed/feature_matrix_3yr.csv"   
MODEL_PATH       = f"{BASE}/processed/rf_model_40yr.joblib"
MAPS_PATH        = f"{BASE}/outputs"

files_and_folders = {
    "SEPA PVA":              SEPA_PATH,
    "OSM Buildings":         OSM_BUILD,
    "OSM Roads":             OSM_ROADS,
    "OSM Water":              OSM_WATER,
    "NASA Elevation":        NASA_ELEV,
    "Rainfall archive 1987-2025": RAIN_ARCHIVE,
    "Rainfall daily parquet": RAIN_DAILY_PARQUET,
    "Rainfall features 40yr": RAIN_FEATURES_40YR,
    "Feature Matrix":        FEATURE_CSV,
    "Feature Matrix (coords)": FEATURE_CSV_COORDS,
    "Feature Matrix (40yr)": FEATURE_CSV_40YR,
    "Feature Matrix (3yr snapshot)": FEATURE_CSV_3YR,
    "RF model (40yr)":       MODEL_PATH,
}

print("Dataset availability check:")
print("=" * 55)
for name, path in files_and_folders.items():
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    if exists and os.path.isdir(path):
        n_files = len([f for f in os.listdir(path) if f.endswith(".nc")])
        print(f"  {name:<28} {status}  ({n_files} .nc files)")
    elif exists:
        size = os.path.getsize(path) / (1024*1024)
        print(f"  {name:<28} {status}  ({size:.1f} MB)")
    else:
        print(f"  {name:<28} {status}")
print("=" * 55)

## Load SEPA Potentially Vulnerable Areas (PVA)  flood boundary zones

In [3]:
df_sepa = gpd.read_file(SEPA_PATH)
df_sepa.head(10)

,OBJECTID_1,PVA_Name,Desig_Date,Chng_C1,PVA_Ref,Shape_Length,Shape_Area,geometry
0,1,Thurso and Halkirk,2018-12-18,Boundary change,02/01/01,84478.694740,1.027316e+08,"MULTIPOLYGON (((308974.999 970125, 308974.999 ..."
1,2,Wick,2018-12-18,Boundary change,02/01/02,100485.288215,8.650975e+07,"MULTIPOLYGON (((334273.843 952200.314, 334261...."
2,3,Lochinver,2018-12-18,Boundary change,02/01/03,57670.830257,4.344988e+07,"MULTIPOLYGON (((209625 923075, 209625 923025, ..."
3,4,Golspie,2011-12-22,No change,02/01/04,36035.606949,2.416975e+07,"MULTIPOLYGON (((283968.105 900005.925, 283962 ..."
4,5,Dornoch,2011-12-22,No change,02/01/05,58566.815141,4.052294e+07,"MULTIPOLYGON (((274774.999 896075, 274775 8960..."
5,6,Aird Point,2018-12-18,New PVA,02/01/06,52469.850623,5.206283e+07,"MULTIPOLYGON (((188555.999 897337.001, 188611 ..."
6,7,Gairloch,2018-12-18,New PVA,02/01/07,96088.344946,7.179356e+07,"MULTIPOLYGON (((180990 875134.999, 181025 8751..."
7,8,Tarbat Ness,2011-12-22,No change,02/01/08,116236.905978,7.762992e+07,"MULTIPOLYGON (((287371 876719, 287339.999 8766..."
8,9,Invergordon,2018-12-18,Boundary change,02/01/09,30046.544683,2.737568e+07,"MULTIPOLYGON (((270875 874925, 270875 874874.9..."
9,10,Alness,2011-12-22,No change,02/01/10,56861.146822,6.002366e+07,"MULTIPOLYGON (((260275 879825.001, 260275 8797..."


## Load OSM buildings

In [4]:
buildings_raw = gpd.read_file(OSM_BUILD)
buildings_raw.head(10)

,element,id,building,building:levels,height,name,geometry
0,node,1495513134,yes,None,None,The Hengler's Circus,POINT (258251.041 665933.24)
1,node,1705790482,yes,None,None,None,POINT (261171.756 663284.757)
2,node,1936303000,hotel,None,None,Fraser Suites,POINT (259568.006 664953.44)
3,node,2180409219,construction,None,None,Next,POINT (259150.854 665012.542)
4,node,2371954470,dormitory,None,None,Queen Margaret Halls,POINT (256448.68 668027.705)
5,node,2452854341,garage,None,None,None,POINT (261017.459 664194.037)
6,node,3218826826,university,None,None,Department of Theology & Religious Studies,POINT (256796.717 666714.266)
7,node,3813079919,commercial,None,None,Esso - Kelvinside,POINT (257520.079 667893.428)
8,node,5001302993,retail,None,None,Holland & Barrett,POINT (258752.96 665252.981)
9,node,5159005096,warehouse,None,None,Mammoet Ferry Transport (UK) BV,POINT (264035.06 661463.622)


## Load OSM roads

In [5]:
roads_raw = gpd.read_file(OSM_ROADS)
roads_raw.head(20)

,element,id,highway,name,geometry
0,node,194815,motorway_junction,Bothwell Street Interchange,POINT (257999.408 665090.116)
1,node,194816,motorway_junction,St George's Cross,POINT (257950.826 666001.954)
2,node,194825,motorway_junction,Townhead Interchange,POINT (260023.156 666227.352)
3,node,194829,motorway_junction,Blochairn,POINT (261052.051 665913.363)
4,node,352856,traffic_signals,None,POINT (264127.527 662433.314)
5,node,352986,motorway_junction,Cumbernauld Road,POINT (263118.393 666681.832)
6,node,352998,motorway_junction,Cumbernauld Road,POINT (263745.392 666389.767)
7,node,352999,motorway_junction,Provan,POINT (261910.505 666100.818)
8,node,353009,motorway_junction,Townhead Interchange,POINT (260495.526 665920.359)
9,node,353017,motorway_junction,St George's Cross,POINT (258487.404 666374.306)


## Load OSM water bodies

In [6]:
water_raw = gpd.read_file(OSM_WATER)
water_raw.head(20)

,element,id,name,geometry
0,relation,919880,Nature Pond,"POLYGON ((257674.475 662316.729, 257677.239 66..."
1,relation,1230156,Hogganfield Loch,"POLYGON ((263898.758 667143.583, 263892.042 66..."
2,relation,1235452,None,"POLYGON ((261119.752 668741.567, 261131.394 66..."
3,relation,1235453,None,"POLYGON ((260913.581 668768.335, 260924.731 66..."
4,relation,1288901,River Clyde,"POLYGON ((268768.649 659687.368, 268781.771 65..."
5,relation,1630110,None,"POLYGON ((261960.785 665669.829, 261968.354 66..."
6,relation,1947036,None,"POLYGON ((262788.359 668120.029, 262802.511 66..."
7,relation,2099637,None,"POLYGON ((260345.266 663042.392, 260344.344 66..."
8,relation,2266953,River Clyde,"POLYGON ((255631.857 665944.366, 255634.096 66..."
9,relation,3118337,None,"POLYGON ((256554.22 662984.379, 256564.851 662..."


## Load NASA SRTM elevation

In [7]:
elevation_raw = gpd.read_file(NASA_ELEV)
elevation_raw.head(20)

,latitude,longitude,elevation,geometry
0,55.92,-4.350000,79,POINT (253242.407 672152.61)
1,55.92,-4.349722,79,POINT (253259.762 672152.02)
2,55.92,-4.349444,79,POINT (253277.118 672151.43)
3,55.92,-4.349167,79,POINT (253294.473 672150.84)
4,55.92,-4.348889,78,POINT (253311.828 672150.251)
5,55.92,-4.348611,78,POINT (253329.183 672149.661)
6,55.92,-4.348333,77,POINT (253346.539 672149.071)
7,55.92,-4.348056,77,POINT (253363.894 672148.482)
8,55.92,-4.347778,78,POINT (253381.249 672147.892)
9,55.92,-4.347500,78,POINT (253398.605 672147.303)


## Load rainfall data (1987-2025)

In [8]:
sample_file = sorted(
    f for f in os.listdir(RAIN_ARCHIVE) if f.endswith(".nc")
)[0]

n_files = len([f for f in os.listdir(RAIN_ARCHIVE) if f.endswith(".nc")])
print(f"Rainfall archive (1987-2025, merged): {n_files} files, e.g. {sample_file}")

ds = nc.Dataset(os.path.join(RAIN_ARCHIVE, sample_file))
print(f"\n=== Sample: {sample_file} ===")
print("Variables:", list(ds.variables.keys()))
print("Dimensions:", {k: len(v) for k, v in ds.dimensions.items()})
print("Rainfall var shape:", ds.variables['rainfall'].shape,
      "fill value:", ds.variables['rainfall']._FillValue)
ds.close()

Rainfall archive (1987-2025, merged): 467 files, e.g. rainfall_hadukgrid_uk_5km_day_19870101-19870131.nc

=== Sample: rainfall_hadukgrid_uk_5km_day_19870101-19870131.nc ===
Variables: ['rainfall', 'transverse_mercator', 'time', 'time_bnds', 'projection_y_coordinate', 'projection_y_coordinate_bnds', 'projection_x_coordinate', 'projection_x_coordinate_bnds', 'latitude', 'longitude']
Dimensions: {'time': 31, 'projection_y_coordinate': 290, 'projection_x_coordinate': 180, 'bnds': 2}
Rainfall var shape: (31, 290, 180) fill value: 1e+20
